# 4. AlphaGenome Output Review

This notebook reviews post-run AlphaGenome outputs across the selected loci inventory.

It does four things:

1. Loads the selected loci inventory used for the AlphaGenome batch run.
2. Merges Phase 1 variant counts with AlphaGenome scoring counts.
3. Summarizes scored variants by TSS-centered vs midpoint-centered window type.
4. Plots per-locus histograms for raw and quantile scores colored by window centering type.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
from statistics import NormalDist

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'utils').is_dir():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / 'utils').is_dir() and (candidate / 'README.md').exists():
            PROJECT_ROOT = candidate
            break

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

from utils.locus_manifest import (
    load_annotation_selection,
    load_loci_manifest,
    merge_annotation_selection,
    validate_annotation_selection,
    validate_loci_manifest,
)
from utils.paths import configure_runtime_env

PATHS = configure_runtime_env(PROJECT_ROOT)
NORM = NormalDist()
RAW_SCALE_FACTOR = 100.0
SCALE_CAP = 2.0
QUANTILE_EPS = 1e-6

sns.set_theme(style='whitegrid', context='notebook')


In [ ]:
MANIFEST_PATH = PROJECT_ROOT / 'config' / 'loci_manifest_sample_100_per_chrom.csv'
SELECTION_PATH = None
SUMMARY_PATH = None

if SELECTION_PATH is None:
    selection_candidates = sorted(
        (PROJECT_ROOT / 'output' / 'prelim' / 'phase1_metrics_screening_review').glob(
            'representative_gene_sample_n*_annotation_selection.csv'
        )
    )
    if not selection_candidates:
        raise FileNotFoundError(
            'No representative_gene_sample_n*_annotation_selection.csv files were found under '
            'output/prelim/phase1_metrics_screening_review.'
        )
    SELECTION_PATH = selection_candidates[-1]
else:
    SELECTION_PATH = Path(SELECTION_PATH)

if SUMMARY_PATH is None:
    summary_candidate = PATHS.output_annotation_alphagenome / f'{SELECTION_PATH.stem}_summary.csv'
    if summary_candidate.exists():
        SUMMARY_PATH = summary_candidate
    else:
        fallback_candidate = PATHS.output_annotation_alphagenome / f'{SELECTION_PATH.stem}.csv'
        legacy_candidate = PATHS.output_annotation_alphagenome / f'{SELECTION_PATH.stem.replace("_annotation_selection", "")}_annotation_selection_summary.csv'
        if fallback_candidate.exists():
            SUMMARY_PATH = fallback_candidate
        elif legacy_candidate.exists():
            SUMMARY_PATH = legacy_candidate
        else:
            raise FileNotFoundError(
                f'Could not find an AlphaGenome summary CSV for selection file {SELECTION_PATH.name} under '
                f'{PATHS.output_annotation_alphagenome}'
            )
else:
    SUMMARY_PATH = Path(SUMMARY_PATH)

selection_stem = SELECTION_PATH.stem
phase1_metrics_candidates = [
    PATHS.output_prelim / f'{selection_stem.replace("_annotation_selection", "_manifest")}_phase1_dataset_metrics.csv',
    PATHS.output_prelim / 'representative_gene_sample_n10_manifest_phase1_dataset_metrics.csv',
    PATHS.output_prelim / 'loci_manifest_sample_100_per_chrom_phase1_dataset_metrics.csv',
]
PHASE1_METRICS_PATH = next((candidate for candidate in phase1_metrics_candidates if candidate.exists()), None)
if PHASE1_METRICS_PATH is None:
    raise FileNotFoundError('Could not resolve a Phase 1 metrics CSV under output/prelim/.')

REVIEW_DIR = PATHS.output_annotation_alphagenome / 'review' / selection_stem
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

print(f'Manifest: {MANIFEST_PATH}')
print(f'Selection: {SELECTION_PATH}')
print(f'AlphaGenome summary: {SUMMARY_PATH}')
print(f'Phase 1 metrics: {PHASE1_METRICS_PATH}')
print(f'Review output dir: {REVIEW_DIR}')

In [ ]:
manifest_df = load_loci_manifest(MANIFEST_PATH)
validate_loci_manifest(manifest_df)
selection_df = load_annotation_selection(SELECTION_PATH)
validate_annotation_selection(selection_df, manifest_df)
selected_loci_df = merge_annotation_selection(manifest_df, selection_df)

phase1_df = pd.read_csv(PHASE1_METRICS_PATH)
summary_df = pd.read_csv(SUMMARY_PATH)

selected_loci_df = selected_loci_df[[
    'locus_id', 'gene_name', 'gene_id', 'gtex_tissue', 'gtex_chrom', 'priority', 'notes'
]].copy()

review_df = selected_loci_df.merge(
    phase1_df[[
        'locus_id',
        'raw_gene_variants_loaded',
        'z_score_variants_exported',
        'ld_matched_variants',
        'phase1_master_variants',
        'status',
        'error_message',
    ]].rename(columns={
        'status': 'phase1_status',
        'error_message': 'phase1_error_message',
    }),
    on='locus_id',
    how='left',
).merge(
    summary_df[[
        'locus_id',
        'status',
        'error_message',
        'variants_loaded',
        'variants_tss_centered',
        'variants_midpoint_centered',
        'variants_ineligible',
        'variants_to_score',
        'variants_attempted',
        'variants_scored_ok',
        'variants_api_error',
        'aggregated_variants',
        'variant_scores_csv',
        'filtered_scores_parquet',
        'variant_window_eligibility_csv',
    ]].rename(columns={
        'status': 'alphagenome_status',
        'error_message': 'alphagenome_error_message',
    }),
    on='locus_id',
    how='left',
)

review_df = review_df.rename(columns={
    'raw_gene_variants_loaded': 'gtex_variant_count_raw',
    'z_score_variants_exported': 'gtex_variant_count_post_filter',
    'ld_matched_variants': 'ld_variant_count_matched',
    'phase1_master_variants': 'ld_variant_count_phase1_master',
    'variants_scored_ok': 'alphagenome_variant_count_scored',
})

review_df = review_df.sort_values(['priority', 'locus_id'], kind='stable').reset_index(drop=True)
review_df.to_csv(REVIEW_DIR / 'alphagenome_locus_review_summary.csv', index=False)
review_df

## Locus Count Summary

For each selected locus:

- `gtex_variant_count_raw`: raw low-side GTEx count loaded for the gene
- `ld_variant_count_phase1_master`: final Phase 1 master variant count after LD matching and export
- `alphagenome_variant_count_scored`: number of variants with aggregated AlphaGenome scores on disk
- `variants_tss_centered` / `variants_midpoint_centered`: scored variants split by windowing mode
- `variants_ineligible`: Phase 1 variants excluded before AlphaGenome scoring


In [ ]:
count_summary_df = review_df[[
    'locus_id',
    'gene_name',
    'gtex_chrom',
    'gtex_variant_count_raw',
    'ld_variant_count_phase1_master',
    'alphagenome_variant_count_scored',
    'variants_tss_centered',
    'variants_midpoint_centered',
    'variants_ineligible',
    'alphagenome_status',
    'alphagenome_error_message',
]].copy()
count_summary_df

In [ ]:
aggregate_counts_df = pd.DataFrame([
    {
        'n_loci_selected': len(review_df),
        'n_loci_completed': int(review_df['alphagenome_status'].astype(str).eq('completed').sum()),
        'gtex_variant_count_raw_total': int(review_df['gtex_variant_count_raw'].fillna(0).sum()),
        'ld_variant_count_phase1_master_total': int(review_df['ld_variant_count_phase1_master'].fillna(0).sum()),
        'alphagenome_variant_count_scored_total': int(review_df['alphagenome_variant_count_scored'].fillna(0).sum()),
        'variants_tss_centered_total': int(review_df['variants_tss_centered'].fillna(0).sum()),
        'variants_midpoint_centered_total': int(review_df['variants_midpoint_centered'].fillna(0).sum()),
        'variants_ineligible_total': int(review_df['variants_ineligible'].fillna(0).sum()),
    }
])
aggregate_counts_df.to_csv(REVIEW_DIR / 'alphagenome_aggregate_count_summary.csv', index=False)
aggregate_counts_df

In [ ]:
def scale_raw_scores(series: pd.Series, factor: float = RAW_SCALE_FACTOR, cap: float = SCALE_CAP) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce')
    return (values * factor).clip(lower=-cap, upper=cap)


def scale_quantile_scores(series: pd.Series, cap: float = SCALE_CAP, eps: float = QUANTILE_EPS) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce').clip(lower=-1 + eps, upper=1 - eps)
    probs = (values + 1.0) / 2.0
    scaled = probs.apply(NORM.inv_cdf)
    return scaled.clip(lower=-cap, upper=cap)


score_rows = []

for locus_row in review_df.itertuples(index=False):
    score_path = Path(locus_row.variant_scores_csv) if pd.notna(locus_row.variant_scores_csv) else None
    eligibility_path = Path(locus_row.variant_window_eligibility_csv) if pd.notna(locus_row.variant_window_eligibility_csv) else None
    phase1_master_path = PATHS.output_ld / locus_row.locus_id / f'{locus_row.gene_name}_phase1_master_variants.csv'
    if score_path is None or eligibility_path is None or not score_path.exists() or not eligibility_path.exists():
        continue
    if not phase1_master_path.exists():
        continue

    score_df = pd.read_csv(score_path)
    eligibility_df = pd.read_csv(eligibility_path)
    phase1_master_df = pd.read_csv(phase1_master_path, usecols=['variant_id', 'z_score'])

    merged_df = (
        score_df
        .merge(
            eligibility_df[['variant_id', 'scoring_mode']].rename(columns={'variant_id': 'source_variant_id'}),
            on='source_variant_id',
            how='left',
        )
        .merge(
            phase1_master_df.rename(columns={'variant_id': 'source_variant_id'}),
            on='source_variant_id',
            how='left',
        )
    )
    merged_df['raw_scaled'] = scale_raw_scores(merged_df['alphagenome_raw_mean'])
    merged_df['quantile_scaled'] = scale_quantile_scores(merged_df['alphagenome_quantile_mean'])
    merged_df['locus_id'] = locus_row.locus_id
    merged_df['gene_name'] = locus_row.gene_name
    score_rows.append(merged_df)

if not score_rows:
    raise ValueError('No AlphaGenome variant score CSVs plus eligibility CSVs could be loaded for plotting.')

variant_scores_df = pd.concat(score_rows, ignore_index=True)
variant_scores_df.to_csv(REVIEW_DIR / 'alphagenome_variant_score_review_table.csv', index=False)
variant_scores_df.head()


In [ ]:
plot_df = variant_scores_df[[
    'locus_id',
    'gene_name',
    'source_variant_id',
    'scoring_mode',
    'z_score',
    'alphagenome_raw_mean',
    'alphagenome_quantile_mean',
    'raw_scaled',
    'quantile_scaled',
]].copy()

plot_df = plot_df.rename(columns={
    'z_score': 'gtex_z_score',
    'alphagenome_raw_mean': 'raw_mean',
    'alphagenome_quantile_mean': 'quantile_mean',
})

plot_long_df = plot_df.melt(
    id_vars=['locus_id', 'gene_name', 'source_variant_id', 'scoring_mode'],
    value_vars=['gtex_z_score', 'raw_mean', 'quantile_mean', 'raw_scaled', 'quantile_scaled'],
    var_name='score_type',
    value_name='score_value',
)
plot_long_df['score_type'] = plot_long_df['score_type'].map({
    'gtex_z_score': 'GTEx Z-score',
    'raw_mean': 'Raw Mean',
    'quantile_mean': 'Quantile Mean',
    'raw_scaled': 'Raw Scaled',
    'quantile_scaled': 'Quantile Scaled',
})
plot_long_df.head()


## Per-Locus Histograms

Each figure below is split by locus.

- rows: GTEx z-score, raw mean, quantile mean, raw scaled, quantile scaled
- colors: `tss_centered` vs `midpoint_centered`
- `raw_scaled = clip(RAW_SCALE_FACTOR * raw_mean, -2, 2)`
- `quantile_scaled = clip(Phi^{-1}((quantile_mean + 1) / 2), -2, 2)` with endpoint clipping to avoid infinities


In [ ]:
plot_paths = []
palette = {
    'tss_centered': '#2f6c8f',
    'midpoint_centered': '#b85c38',
}
score_order = ['GTEx Z-score', 'Raw Mean', 'Quantile Mean', 'Raw Scaled', 'Quantile Scaled']

for locus_row in review_df[['locus_id', 'gene_name']].drop_duplicates().itertuples(index=False):
    locus_plot_df = plot_long_df[plot_long_df['locus_id'] == locus_row.locus_id].copy()
    if locus_plot_df.empty:
        continue

    fig, axes = plt.subplots(len(score_order), 1, figsize=(10, 18), sharex=False)
    for ax, score_type in zip(axes, score_order):
        subset_df = locus_plot_df[locus_plot_df['score_type'] == score_type].copy()
        subset_df = subset_df.dropna(subset=['score_value'])
        if subset_df.empty:
            ax.set_visible(False)
            continue
        for scoring_mode in ['tss_centered', 'midpoint_centered']:
            mode_df = subset_df[subset_df['scoring_mode'] == scoring_mode]
            if mode_df.empty:
                continue
            ax.hist(
                mode_df['score_value'],
                bins=40,
                alpha=0.65,
                label=scoring_mode,
                color=palette.get(scoring_mode, '#666666'),
                edgecolor='white',
            )
        ax.set_title(f'{locus_row.gene_name} ({locus_row.locus_id}) :: {score_type}')
        ax.set_xlabel(score_type)
        ax.set_ylabel('Variant count')
        ax.legend(loc='best')

    fig.tight_layout()
    plot_path = REVIEW_DIR / f'{locus_row.gene_name}_{locus_row.locus_id}_score_histograms.png'
    fig.savefig(plot_path, dpi=150)
    plt.show()
    plt.close(fig)
    plot_paths.append({'locus_id': locus_row.locus_id, 'gene_name': locus_row.gene_name, 'plot_path': str(plot_path)})

plot_manifest_df = pd.DataFrame(plot_paths)
plot_manifest_df.to_csv(REVIEW_DIR / 'alphagenome_review_plot_manifest.csv', index=False)
plot_manifest_df
